# OULAD → Neo4j Aura: ETL test run

Runs the loading pipeline end to end from Google Colab and checks the result, using
Colab's secret store for credentials.

**Before you start**, open the key icon in the left sidebar and add these secrets, switching
on *Notebook access* for each one:

| Secret | Required | Notes |
| --- | --- | --- |
| `NEO4J_URI` | yes | e.g. `neo4j+s://xxxxxxxx.databases.neo4j.io` |
| `NEO4J_USERNAME` | yes | usually `neo4j` |
| `NEO4J_PASSWORD` | yes | from the Aura console |
| `NEO4J_DATABASE` | no | defaults to the driver's database |

The pipeline writes to whatever instance those point at. It is safe to re-run: node loads
`MERGE`, and relationship loads are guarded so a second pass creates nothing.

Expect roughly **7 minutes** against an empty database, or **4½** re-running against a
loaded one. Most of that is the 8.46M `REVIEWED_MATERIAL` relationships.

## 1. Get the code

Re-running this cell updates an existing clone instead of failing.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/jose-alvarado-guzman/oulad.git'
REPO_DIR = '/content/oulad'

def run(*command, cwd=None):
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run('git', '-C', REPO_DIR, 'pull', '--ff-only')
else:
    run('git', 'clone', '--depth', '1', REPO_URL, REPO_DIR)

run('git', '-C', REPO_DIR, 'log', '-1', '--oneline')

## 2. Install the dependencies

**A session restart is required here, not optional.** Colab pins
`traitlets==5.7.1`, and `pyneoinstance` cannot be imported against it: it pulls in
`neo4j-viz`, whose widget module evaluates `traitlets.Instance[...]` while defining a class,
and subscripting `Instance` only works from traitlets 5.10. `requirements.txt` therefore
carries its own `traitlets>=5.10` floor, because `neo4j-viz` asks for `traitlets>=5,<6`,
which 5.7.1 already satisfies — so without the floor pip would leave it alone and the import
would fail with `type 'Instance' is not subscriptable`.

The kernel has already imported traitlets 5.7.1 at startup, so pip writing new files on disk
changes nothing until you restart.

After this cell finishes: **Runtime → Restart session**, then continue from step 3. The
clone from step 1 survives the restart. Step 3 verifies the upgrade actually took effect.

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

print('\nInstalled. Now use Runtime -> Restart session, then run step 3.')

## 3. Point Python at the package

Start here again after restarting the session. This cell refuses to continue if the traitlets
upgrade has not taken effect, since the alternative is a confusing failure inside a
third-party import.

In [ ]:
import os
import sys

REPO_DIR = '/content/oulad'
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import traitlets

try:
    traitlets.Instance[int]
except TypeError:
    raise SystemExit(
        f'traitlets {traitlets.__version__} is still loaded, and importing '
        'pyneoinstance against it fails with "type \'Instance\' is not '
        'subscriptable".\n\nStep 2 installed a newer traitlets, but this kernel '
        'is still holding the old module. Use Runtime -> Restart session, then '
        're-run this cell.'
    )

import pandas
import neo4j
import pyneoinstance

print('python        ', sys.version.split()[0])
print('traitlets     ', traitlets.__version__, '(needs >= 5.10)')
print('pandas        ', pandas.__version__)
print('neo4j driver  ', neo4j.__version__)
print('pyneoinstance ', pyneoinstance.__version__)

## 4. Resolve the credentials

`oulad.credentials` reads Colab's secret store when it is available and falls back to a
`.env` file when running locally. A fresh clone has no `.env`, so this cell is a direct test
of the Colab path.

Only the ETL group is required here. The Aura Graph Analytics keys (`CLIENT_ID`,
`CLIENT_SECRET`, `PROJECT_ID`) belong to a separate group that the loading pipeline never
reads — see the last cell.

In [ ]:
from oulad.credentials import (
    AGA_SECRETS,
    ETL_SECRETS,
    MissingCredentialsError,
    in_colab,
    load_credentials,
)
from oulad.logger import get_logger

logger = get_logger(REPO_DIR)

print('Colab secret store available:', in_colab())
try:
    source = load_credentials(logger)
    print('resolved from:', source)
except MissingCredentialsError as error:
    raise SystemExit(f'\n{error}\n\nAdd the missing secrets in the sidebar, '
                     'switch on Notebook access, then re-run this cell.')

print('\nETL group resolved:', all(os.getenv(name) for name in ETL_SECRETS))
print('database          :', os.getenv('NEO4J_DATABASE') or "(driver's default)")

## 5. Run the ETL

**In this kernel, not as `!python -m oulad`.** Colab's secret store is only reachable from
the kernel process: a subprocess can import `google.colab`, but its `userdata.get` call has
no channel back to the notebook, so credential resolution would fall through to a `.env`
file that a fresh clone does not have.

The pipeline connects and creates its constraints *before* downloading anything, so a wrong
password fails in seconds rather than after the 44.6 MiB download. Progress is logged here
and to `Logs/import.log`.

In [ ]:
from oulad.__main__ import main

try:
    main()
except SystemExit as stop:
    raise RuntimeError(
        'The pipeline exited early; the logged error above says why.') from stop

## 6. Read the quality-check tables

Both load phases write a timestamped csv to `Result/`. `qaFlag` is `toLoad - postCount`, so
**every row should read 0**.

Which column carries the totals tells you what kind of run this was: on a first load against
an empty database `nodesCreated` holds them and `priorNodeCount` is 0, and on a re-run the
two swap.

In [ ]:
import glob

import pandas as pd
from IPython.display import display

for pattern in ('node_qa_results_*.csv', 'relationships_qa_results_*.csv'):
    matches = glob.glob(os.path.join(REPO_DIR, 'Result', pattern))
    newest = max(matches, key=os.path.getmtime)
    frame = pd.read_csv(newest)
    print(os.path.basename(newest))
    display(frame)
    flagged = frame[frame.qaFlag != 0]
    print('rows with a non-zero qaFlag:', len(flagged), '\n')

## 7. Check the graph independently

The QA tables come from the pipeline's own bookkeeping, so these queries go and look at the
graph instead. Against the full dataset expect **66,920 nodes** and **8,818,076
relationships**.

The last two are model invariants rather than totals: every student belongs to exactly one
age group, which is what the `first-by` rule in `config.yaml` exists to guarantee, and the
`DisabledStudent` label should sit on 2,717 students.

In [ ]:
from pyneoinstance import Neo4jInstance

database = os.getenv('NEO4J_DATABASE') or None
graph = Neo4jInstance(
    os.getenv('NEO4J_URI'),
    os.getenv('NEO4J_USERNAME'),
    os.getenv('NEO4J_PASSWORD'),
)
ask = lambda cypher: graph.execute_read_query(cypher, database=database)

print('nodes        ', f"{ask('MATCH (n) RETURN count(n) AS c').c.item():,}", '(expect 66,920)')
print('relationships', f"{ask('MATCH ()-[r]->() RETURN count(r) AS c').c.item():,}", '(expect 8,818,076)')
print('constraints  ', len(graph.get_constraints(database=database)), '(expect 9)')

print('\nnode labels')
display(graph.get_node_label_freq(database=database))
print('relationship types')
display(graph.get_rela_type_freq(database=database))

extra_age_groups = ask('''
    MATCH (s:Student)-[:IN_AGE_GROUP]->()
    WITH s, count(*) AS groups
    WHERE groups > 1
    RETURN count(s) AS c
''').c.item()
disabled = ask('MATCH (n:DisabledStudent) RETURN count(n) AS c').c.item()
print('\nstudents in more than one age group:', extra_age_groups, '(expect 0)')
print('DisabledStudent nodes              :', f'{disabled:,}', '(expect 2,717)')

## Next: Aura Graph Analytics

The analytics that run against this graph need a different set of secrets from the loader,
so they are enforced separately. Nothing above required them, which is why a load can
succeed in an environment that is not yet set up for analytics.

The cell below only reports what is present; it does not fail on missing keys.

In [ ]:
print('AGA secret group:', AGA_SECRETS)
for name in AGA_SECRETS:
    print(f'  {name:15s}', 'set' if os.getenv(name) else 'not set')

print('\nAnalytics code asks for its own group like this:')
print('    load_credentials(logger, required=AGA_SECRETS)')
print('\nAdd those three as Colab secrets when you start on the analytics.')